# 10.1 General principles of commands and options

In [1]:
# install dependencies
%pip install -q amplpy pandas

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

With amplpy, an AMPL session is started from Python by creating an `AMPL` object. In a notebook such as this one, the `ampl_notebook()` call in the first cell does this for you: it installs the requested solver modules, activates the license, and returns an `AMPL` object, which we call `ampl` throughout this book. In a standalone Python script, you create the object yourself:

```python
from amplpy import AMPL

ampl = AMPL()      # start an AMPL session
...            # commands for the session
ampl.close()   # end the session
```

The installation and license setup necessarily vary somewhat from one environment to another; for details, you should refer to the [amplpy documentation](https://amplpy.ampl.com/en/latest/getting-started.html).

## 10.1.1 Commands

Each AMPL command is given by calling a method of the `AMPL` object; AMPL processes the command, prints any appropriate messages in response, and returns control to Python. The most frequently used commands have methods of their own, such as `read()`, `solve()` and `display()`. Any other AMPL command can be passed as a string to the `eval()` method. The `get_output()` method works the same way, but returns the output of the commands as a Python string instead of printing it. Whenever a command has a method of its own, the method should be preferred; we use `eval()` only for the few commands that have none.

```{admonition} Curiosity: AMPL's command prompt
:class: tip

In the old days of the AMPL IDE and of AMPL without APIs, after starting AMPL, the first thing you see is AMPL's prompt, `ampl:`. Whenever you see this prompt, AMPL is ready to read and interpret what you type, and it processes everything you typed on the line when you press the "enter" or "return" key. A command ends with a semicolon; if you end a line in the middle of a command, AMPL prompts you to continue it on the next line, with a prompt that ends with a question mark: `ampl?`.

If you have used AMPL this way, the amplpy methods for the most frequently used commands will look familiar:

| AMPL command | amplpy method |
| :--- | :--- |
| `model diet.mod;` | `ampl.read("diet.mod")` |
| `data diet.dat;` | `ampl.read_data("diet.dat")` |
| `solve;` | `ampl.solve()` |
| `display Buy;` | `ampl.display("Buy")` |
| `option display_width 60;` | `ampl.option["display_width"] = 60` |
| `reset;` | `ampl.reset()` |
| `end;` or `quit;` | `ampl.close()` |
```

To illustrate, we load the diet model of [Chapter 2](../02/02.md), and set its data from Pandas DataFrames as described in [Section 9.4](../09/9_4_data_in_pandas.ipynb):

In [2]:
%%writefile diet.mod

set NUTR;
set FOOD;

param cost {FOOD} > 0;
param f_min {FOOD} >= 0;
param f_max {j in FOOD} >= f_min[j];

param n_min {NUTR} >= 0;
param n_max {i in NUTR} >= n_min[i];

param amt {NUTR,FOOD} >= 0;

var Buy {j in FOOD} >= f_min[j], <= f_max[j];

minimize Total_Cost:  sum {j in FOOD} cost[j] * Buy[j];

subject to Diet {i in NUTR}:
   n_min[i] <= sum {j in FOOD} amt[i,j] * Buy[j] <= n_max[i];

Writing diet.mod


In [3]:
import pandas as pd

ampl.read("diet.mod")

food_df = pd.DataFrame(
    [
        ("BEEF", 3.19, 2, 10),
        ("CHK",  2.59, 2, 10),
        ("FISH", 2.29, 2, 10),
        ("HAM",  2.89, 2, 10),
        ("MCH",  1.89, 2, 10),
        ("MTL",  1.99, 2, 10),
        ("SPG",  1.99, 2, 10),
        ("TUR",  2.49, 2, 10),
    ],
    columns=["FOOD", "cost", "f_min", "f_max"],
).set_index("FOOD")

nutr_df = pd.DataFrame(
    [
        ("A",     700, 20000),
        ("C",     700, 20000),
        ("B1",    700, 20000),
        ("B2",    700, 20000),
        ("NA",      0, 50000),
        ("CAL", 16000, 24000),
    ],
    columns=["NUTR", "n_min", "n_max"],
).set_index("NUTR")

# Nutrient content: one row per nutrient, one column per food
amt_df = pd.DataFrame(
    {
        "BEEF": [60, 20, 10, 15,  938, 295],
        "CHK":  [ 8,  0, 20, 20, 2180, 770],
        "FISH": [ 8, 10, 15, 10,  945, 440],
        "HAM":  [40, 40, 35, 10,  278, 430],
        "MCH":  [15, 35, 15, 15, 1182, 315],
        "MTL":  [70, 30, 15, 15,  896, 400],
        "SPG":  [25, 50, 25, 15, 1329, 370],
        "TUR":  [60, 20, 15, 10, 1397, 450],
    },
    index=["A", "C", "B1", "B2", "NA", "CAL"],
)

ampl.set_data(food_df, "FOOD")             # set FOOD and parameters cost, f_min, f_max
ampl.set_data(nutr_df, "NUTR")             # set NUTR and parameters n_min, n_max
ampl.param["amt"] = amt_df.stack().to_dict()   # {(nutrient, food): amount}

The `display()` method takes the same AMPL expressions as the `display` command, written as a Python string. For example, to display all nutrient amounts of at least 500:

In [4]:
ampl.display("{i in NUTR, j in FOOD: amt[i,j] >= 500} amt[i,j]")

amt[i,j] :=
CAL CHK     770
NA  BEEF    938
NA  CHK    2180
NA  FISH    945
NA  MCH    1182
NA  MTL     896
NA  SPG    1329
NA  TUR    1397
;



A string passed to `eval()` or `get_output()` must contain complete AMPL commands, each ended by a semicolon. Methods that correspond to a single command, such as `display()` and `read()`, supply the semicolon themselves.

Several commands use filenames for reading or writing information. In amplpy, a filename is passed as a Python string. The rules for correct filenames are determined by the operating system, however, not by AMPL. For the examples in this book we have used filenames like `diet.mod` that are acceptable to almost any operating system. A filename that does not begin with a directory is looked up in AMPL's current working directory, which `ampl.cd()` returns, or changes when given a new directory as its argument.

To conclude an AMPL session, call `ampl.close()`. In a notebook, the session also ends when the notebook's kernel is shut down or restarted.

Although amplpy hides AMPL's command interface, it is being used behind the scenes by every amplpy method, so it's well worth understanding the AMPL commands and how to use them effectively.

## 10.1.2 Options

The behavior of AMPL commands depends not only on what you type directly, but on a variety of options for choosing alternative solvers, controlling the display of results, and the like.

Each option has a name, and a value that may be a number or a character string. For example, the option `solver` is a string that names the solver to be used by the `solve` command. The option `display_width` has a numeric value, which says how many characters wide the output produced by the `display` command may be.

The `option` property of the `AMPL` object displays and sets option values. When it is indexed by an option name, amplpy replies with the current value, as a Python number or string:

In [5]:
ampl.option["display_width"], ampl.option["display_precision"]

(79, 6)

The AMPL `option` command can also be given through `get_output()`, which lets us use a `*` in an option name as a "wild card" that matches any sequence of characters:

In [6]:
print(ampl.get_output("option display*;"))

option display_1col 20;
option display_eps 0;
option display_max_2d_cols 0;
option display_precision 6;
option display_round '';
option display_set_1col 0;
option display_transpose 0;
option display_width 79;



The command `option *`, or just `option` alone, lists all current options and values.

When an option name is assigned a value, the named option is reset to the specified value. In the following example we change the display width, and then verify that it has been changed:

In [7]:
ampl.option["display_width"] = 60
ampl.option["display_width"]

60

The methods `ampl.set_option(name, value)` and `ampl.get_option(name)` are equivalent to assigning and reading `ampl.option[name]`.

String values are written as ordinary Python strings, so the quoting rules are those of Python rather than AMPL. An empty string `""` is a meaningful value for some options. At the other extreme, a long string can be spread over several lines, or built up piece by piece, with Python's usual string operations.

When AMPL starts, it sets many options to initial, or default, values. The `display_width` option has a default value of 79, for instance. Other options, especially ones that pertain to particular solvers, are initially unset, and amplpy returns `None` for them:

In [8]:
print(ampl.option["highs_options"])

None


To return all options to their default values, use the command `reset options`:

In [9]:
ampl.eval("reset options;")
ampl.option["display_width"]

79

AMPL maintains no master list of valid options, but rather accepts any new option that you define. Thus if you mis-type an option name, you will most likely define a new option by mistake, as the following example demonstrates:

In [10]:
ampl.option["display_wdith"] = 60   # misspelled option name
print(ampl.get_output("option display_w*;"))

option display_wdith 60;
option display_width 79;



Assigning an option also doesn't check to see if you have given it a meaningful value. You will be informed of a value error only when an option is used by some subsequent command. In these respects, AMPL options are much like operating system or shell "environment variables." In fact you can use the settings of environment variables to override AMPL's option defaults; see your system-specific documentation for details.